In [ ]:
import os
import numpy as np
try:
    import rasterio as rio
    from rasterio.plot import show
except Exception as e:
    print('Rasterio not available:', e)
try:
    import matplotlib.pyplot as plt
except Exception as e:
    print('Matplotlib not available:', e)

## Paths
Set `LABELS_TIF` and `PRED_TIF`. Defaults point to synthetic demo outputs.

In [ ]:
LABELS_TIF = os.environ.get('LABELS_TIF', 'data/benchmark/labels.tif')
PRED_TIF = os.environ.get('PRED_TIF', 'data/benchmark/preds_mock.tif')
print('Labels:', LABELS_TIF)
print('Preds: ', PRED_TIF)

## Load rasters

In [ ]:
with rio.open(LABELS_TIF) as lsrc:
    labels = lsrc.read(1)
    lprofile = lsrc.profile
with rio.open(PRED_TIF) as psrc:
    preds = psrc.read(1)
classes = sorted(list(set(np.unique(labels)).union(set(np.unique(preds)))))
print('Classes:', classes)
print('Shapes:', labels.shape, preds.shape)

## Metrics: accuracy and mIoU
Ignored class id can be set to -1 or a specific value if needed.

In [ ]:
def confusion_matrix(y_true, y_pred, classes):
    k = len(classes)
    cm = np.zeros((k, k), dtype=np.int64)
    idx = {c:i for i,c in enumerate(classes)}
    for t,p in zip(y_true.ravel(), y_pred.ravel()):
        if t in idx and p in idx:
            cm[idx[t], idx[p]] += 1
    return cm
def per_class_iou(cm):
    ious = []
    for i in range(cm.shape[0]):
        tp = cm[i,i]
        fp = cm[:,i].sum() - tp
        fn = cm[i,:].sum() - tp
        denom = (tp + fp + fn)
        ious.append(tp / denom if denom > 0 else np.nan)
    return np.array(ious)
cm = confusion_matrix(labels, preds, classes)
acc = np.trace(cm) / cm.sum()
iou = per_class_iou(cm)
miou = np.nanmean(iou)
print('Accuracy:', round(float(acc), 4))
print('mIoU:', round(float(miou), 4))
print('Per-class IoU:', {c: round(float(v),4) for c,v in zip(classes, iou)})

## Visualization

In [ ]:
plt.figure(figsize=(10,4))
plt.subplot(1,2,1)
plt.title('Labels')
show(labels, cmap='tab20')
plt.subplot(1,2,2)
plt.title('Predictions')
show(preds, cmap='tab20')
plt.tight_layout()
plt.show()